# ComfyUI + LTX 2.3 GGUF
Base: pogscafe (2202) + Lightricks/ComfyUI-LTXVideo (3.9k)
---

In [ ]:
BACKEND_NAME = "kaggle-a"
GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"
print(f"Backend: {BACKEND_NAME}")

## 1. Environment

In [ ]:
%%time
import os, sys, subprocess, threading, time, json, requests
from datetime import datetime

home = '/kaggle/working'
COMFY = f'{home}/ComfyUI'
os.chdir(home)

# Pin ComfyUI - tránh update breaking
COMFY_COMMIT = '7fc3ccdcc2fb1f20c4b7dd4aca374db952fd66df'

# virtualenv + python3.10 (pogscafe)
!pip install -q virtualenv
VENV = f'{home}/venv'
if not os.path.exists(VENV):
    !virtualenv {VENV} -p $(which python3.10)
    if not os.path.exists(f'{VENV}/bin/python3.10'):
        !cp /usr/bin/python3.10 {VENV}/bin/
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python
    !ln -sf {VENV}/bin/python3.10 {VENV}/bin/python3

python = f'{VENV}/bin/python'
pip = f'{VENV}/bin/pip'
print(f'OK')

In [ ]:
%%time
# Clone + pin ComfyUI
if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir(COMFY)
!git checkout {COMFY_COMMIT}
!{pip} install -q -r requirements.txt
print('Ready')

In [ ]:
%%time
# Custom nodes
os.chdir(f'{COMFY}/custom_nodes')
for u,n in [
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git','ComfyUI-LTXVideo'),
    ('https://github.com/logtd/ComfyUI-LTXTricks.git','ComfyUI-LTXTricks'),
    ('https://github.com/city96/ComfyUI-GGUF.git','ComfyUI-GGUF'),
    ('https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Manager'),
]:
    if not os.path.exists(n):
        !git clone {u}
        print(f'  {n}')
print('Done')

## 2. Models (symlink /tmp)

In [ ]:
%%time
!mkdir -p /tmp/models/{unet,clip,vae}
for d in ['unet','clip','vae']:
    src = f'{COMFY}/models/{d}'
    dst = f'/tmp/models/{d}'
    if os.path.islink(src) or os.path.exists(src):
        !rm -rf {src}
    !ln -sf {dst} {src}
print('Symlinks ready')

In [ ]:
%%time
# LTX-2.3 GGUF Q2_K - ONLY format fit T4 16GB
os.chdir('/tmp/models/unet')
f = 'LTX-2.3-22B-distilled-1.1-Q2_K.gguf'
if not os.path.exists(f):
    !wget -c 'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/LTX-2.3-22B-distilled-1.1-Q2_K.gguf' -O '{f}'
sz = os.path.getsize(f)/1e9
print(f'Model: {sz:.1f} GB')

In [ ]:
%%time
# Text encoder + VAE
for d,f,s in [
    ('clip','gemma-2b.safetensors','text_encoder/model.safetensors'),
    ('vae','ltx-vae.safetensors','vae/vae.safetensors'),
]:
    fp = f'/tmp/models/{d}/{f}'
    if not os.path.exists(fp):
        !wget -c 'https://huggingface.co/Lightricks/LTX-2/resolve/main/{s}' -O '{fp}'
print('Done')

## 3. Start ComfyUI + Tunnel

In [ ]:
# Download pinggy wrapper
!wget -q https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /kaggle/working/pinggy.py

# Start ComfyUI in background
os.chdir(COMFY)
!pkill -f main.py 2>/dev/null
time.sleep(2)
proc = subprocess.Popen([python,'main.py','--headless','--port','8188','--listen','127.0.0.1','--highvram'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for ComfyUI
for i in range(40):
    time.sleep(3)
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'ComfyUI PID: {proc.pid}')
            break
    except:
        pass
print('ComfyUI started')

In [ ]:
# Tunnel Pinggy - bg thread
TUNNEL_URL = None

def tunnel():
    global TUNNEL_URL
    # pinggy.py start ComfyUI - we already did that, just tunnel
    p = subprocess.Popen(['ssh','-p','443','-R0:localhost:8188','a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in p.stdout:
        if 'https://' in line:
            i = line.find('https://')
            TUNNEL_URL = line[i:].strip().split()[0]
            open(f'{home}/url.txt','w').write(TUNNEL_URL)
            print(f'URL: {TUNNEL_URL}')
            break

threading.Thread(target=tunnel, daemon=True).start()
time.sleep(15)

for i in range(30):
    if TUNNEL_URL: break
    try:
        TUNNEL_URL = open(f'{home}/url.txt').read().strip()
    except: pass
    time.sleep(5)

print(f'Tunnel: {TUNNEL_URL}')

## 4. Push URL to Gist

In [ ]:
def gist(url, st):
    d = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            d = json.loads(c) if c.strip() else {}
    except: pass
    d[BACKEND_NAME] = {'url':url,'status':st,'updated':datetime.now().isoformat(),'capabilities':['t2v','i2v','v2v'],'gpu':'t4'}
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files':{'kaggle_backends.json':{'content':json.dumps(d,indent=2)}}}, timeout=10)
    return r.status_code

if TUNNEL_URL:
    print(f'Gist: {gist(TUNNEL_URL,"online")}')
    print(f'POST {TUNNEL_URL}/prompt')
else:
    print('No URL')

## 5. Keep Alive

In [ ]:
try:
    while True:
        time.sleep(300)
        if TUNNEL_URL: gist(TUNNEL_URL,'online')
        print('.',end='')
except:
    if TUNNEL_URL: gist(TUNNEL_URL,'offline')